# Exploration des données IoT — INNOV/CCNB 2026

Ce notebook explore les données du système d'alertes IoT :
- Distribution des valeurs par capteur
- Timeline des alertes
- Heatmap des anomalies
- Corrélation entre capteurs
- Détection par Z-Score rolling
- Comparaison des détecteurs

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print('seaborn non installé — heatmap ignorée')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Connexion à la base SQLite
DB_PATH = '../alerts.db'
conn = sqlite3.connect(DB_PATH)
print('Base connectée :', DB_PATH)

## 1. Chargement des données

In [ ]:
# Lectures capteurs
logs = pd.read_sql('SELECT * FROM sensor_logs ORDER BY created_at', conn,
                   parse_dates=['created_at'])
# Alertes
alerts = pd.read_sql('SELECT * FROM alerts ORDER BY created_at', conn,
                     parse_dates=['created_at', 'resolved_at'])

print(f'Lectures : {len(logs):,} lignes')
print(f'Alertes  : {len(alerts):,} lignes')
print()
print('Lectures par capteur :')
print(logs.groupby('sensor')['value'].describe().round(2))

## 2. Distribution des valeurs par capteur

In [ ]:
sensors  = ['temperature', 'turbidity', 'ph']
colors   = {'temperature': '#e74c3c', 'turbidity': '#3498db', 'ph': '#27ae60'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, sensor in zip(axes, sensors):
    data = logs[logs['sensor'] == sensor]['value']
    if data.empty:
        ax.set_title(f'{sensor} (aucune donnée)')
        continue
    ax.hist(data, bins=30, color=colors[sensor], alpha=0.7, edgecolor='white')
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Moy={data.mean():.2f}')
    ax.set_title(f'Distribution {sensor}', fontweight='bold')
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Fréquence')
    ax.legend(fontsize=9)

plt.suptitle('Distribution des valeurs par capteur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Timeline des alertes

In [ ]:
if alerts.empty:
    print('Aucune alerte dans la base.')
else:
    fig, ax = plt.subplots(figsize=(15, 5))
    level_colors = {'CRITICAL': '#e74c3c', 'WARNING': '#f39c12', 'NORMAL': '#27ae60'}
    sensor_y     = {'temperature': 3, 'turbidity': 2, 'ph': 1}

    for _, row in alerts.iterrows():
        y   = sensor_y.get(row['sensor'], 0)
        col = level_colors.get(row['level'], '#999')
        ax.scatter(row['created_at'], y, color=col, s=30, alpha=0.7, zorder=2)

    ax.set_yticks([1, 2, 3])
    ax.set_yticklabels(['pH', 'Turbidité', 'Température'])
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.xticks(rotation=30, ha='right')

    from matplotlib.lines import Line2D
    legend_els = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=l)
                  for l, c in level_colors.items()]
    ax.legend(handles=legend_els, loc='upper right')
    ax.set_title('Timeline des alertes par capteur et niveau', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Heatmap des anomalies (heure × jour)

In [ ]:
anomalies = logs[logs['level'].isin(['WARNING', 'CRITICAL'])].copy()
if anomalies.empty:
    print('Aucune anomalie disponible pour la heatmap.')
elif not HAS_SEABORN:
    print('seaborn requis pour la heatmap — pip install seaborn')
else:
    anomalies['hour']    = anomalies['created_at'].dt.hour
    anomalies['weekday'] = anomalies['created_at'].dt.day_name()

    pivot = anomalies.pivot_table(
        index='weekday', columns='hour', values='value', aggfunc='count', fill_value=0
    )
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot = pivot.reindex([d for d in day_order if d in pivot.index])

    fig, ax = plt.subplots(figsize=(15, 4))
    sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='d', linewidths=0.5,
                cbar_kws={'label': 'Anomalies'}, ax=ax)
    ax.set_title('Heatmap des anomalies : heure × jour', fontweight='bold')
    ax.set_xlabel('Heure')
    ax.set_ylabel('Jour')
    plt.tight_layout()
    plt.show()

## 5. Corrélation entre capteurs

In [ ]:
# Pivoter les logs pour avoir une colonne par capteur
logs['bucket'] = (logs['created_at'].astype(np.int64) // 10**9 // 5)  # bucket 5s
pivot_corr = logs.pivot_table(index='bucket', columns='sensor', values='value', aggfunc='mean').dropna()

if pivot_corr.shape[0] < 10:
    print('Pas assez de données alignées pour calculer les corrélations.')
else:
    corr = pivot_corr.corr()
    print('Matrice de corrélation de Pearson :')
    print(corr.round(3))
    print()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    pairs = [('temperature','turbidity'), ('temperature','ph'), ('turbidity','ph')]
    pair_colors = ['#e74c3c', '#9b59b6', '#3498db']

    for ax, (x, y), col in zip(axes, pairs, pair_colors):
        if x not in pivot_corr or y not in pivot_corr:
            continue
        ax.scatter(pivot_corr[x], pivot_corr[y], alpha=0.5, color=col, s=15)
        r_val = pivot_corr[[x, y]].corr().iloc[0, 1]
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.set_title(f'{x} vs {y}\nr = {r_val:.3f}', fontweight='bold')

    plt.suptitle('Corrélation entre capteurs', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Détection de dérive (Z-Score rolling)

In [ ]:
WINDOW = 30

fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

for ax, sensor in zip(axes, sensors):
    data = logs[logs['sensor'] == sensor].sort_values('created_at').copy()
    if len(data) < WINDOW + 5:
        ax.set_title(f'{sensor} — pas assez de données')
        continue

    data['roll_mean'] = data['value'].rolling(WINDOW, min_periods=5).mean()
    data['roll_std']  = data['value'].rolling(WINDOW, min_periods=5).std()
    data['z_score']   = (data['value'] - data['roll_mean']) / data['roll_std'].replace(0, np.nan)

    ax.plot(data['created_at'], data['z_score'], linewidth=0.8, color=colors[sensor], label='Z-Score')
    ax.axhline(2,  color='#f39c12', linestyle='--', linewidth=1, label='+2σ')
    ax.axhline(-2, color='#f39c12', linestyle='--', linewidth=1)
    ax.axhline(3,  color='#e74c3c', linestyle='--', linewidth=1, label='+3σ')
    ax.axhline(-3, color='#e74c3c', linestyle='--', linewidth=1)
    ax.axhline(0,  color='#27ae60', linestyle='-',  linewidth=0.5)

    # Marquer les violations
    crit = data[data['z_score'].abs() > 3]
    ax.scatter(crit['created_at'], crit['z_score'], color='#e74c3c', s=25, zorder=5)

    ax.set_ylabel(f'{sensor}\nZ-Score')
    ax.set_title(f'{sensor} — Z-Score rolling (fenêtre={WINDOW})', fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(-6, 6)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
plt.xticks(rotation=30, ha='right')
plt.suptitle('Courbe de dérive par Z-Score rolling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Comparaison des détecteurs (F1 par capteur)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

try:
    import numpy as np
    from src.simulator.generator import IoTSimulator
    from src.detection.rules import RuleBasedDetector
    from src.detection.statistical import ZScoreDetector, IQRDetector, IsolationForestDetector

    WARMUP = 100
    TEST   = 200
    SEED   = 42

    sim = IoTSimulator(seed=SEED)
    readings = []
    while len(readings) < WARMUP + TEST:
        readings.extend(sim.read_all())
    readings = readings[:WARMUP + TEST]

    detectors = {
        'Règles métier': RuleBasedDetector(),
        'Z-Score':       ZScoreDetector(window_size=30),
        'IQR':           IQRDetector(window_size=30),
        'Isolation Forest': IsolationForestDetector(window_size=100, min_samples=20),
    }

    for r in readings[:WARMUP]:
        for det in detectors.values():
            det.analyze(r)

    f1_scores = {}
    for name, det in detectors.items():
        tp = fp = fn = tn = 0
        for r in readings[WARMUP:]:
            true_anom = r.scenario != 'normal'
            pred_anom = det.analyze(r).is_anomaly()
            if   true_anom and pred_anom:     tp += 1
            elif not true_anom and pred_anom: fp += 1
            elif true_anom and not pred_anom: fn += 1
            else:                             tn += 1
        p  = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        r_ = tp / (tp + fn) if (tp + fn) > 0 else 1.0
        f1 = 2 * p * r_ / (p + r_) if (p + r_) > 0 else 0.0
        f1_scores[name] = {'precision': round(p, 3), 'recall': round(r_, 3), 'f1': round(f1, 3)}
        print(f'{name:20s} — P={p:.3f}  R={r_:.3f}  F1={f1:.3f}')

    fig, ax = plt.subplots(figsize=(10, 5))
    names  = list(f1_scores.keys())
    prec   = [f1_scores[n]['precision'] for n in names]
    recall = [f1_scores[n]['recall']    for n in names]
    f1     = [f1_scores[n]['f1']        for n in names]

    x = np.arange(len(names))
    w = 0.25
    ax.bar(x - w, prec,   w, label='Précision', color='#3498db', alpha=0.85)
    ax.bar(x,     recall, w, label='Rappel',    color='#27ae60', alpha=0.85)
    ax.bar(x + w, f1,     w, label='F1-Score',  color='#e74c3c', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Comparaison des détecteurs — Précision / Rappel / F1', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

except ImportError as e:
    print(f'Import impossible : {e}. Lancez ce notebook depuis le dossier notebooks/ du projet.')

In [ ]:
conn.close()
print('Notebook terminé.')